In [1]:
# ============================================================
#   Find every AGU26 presentation by a list of named PIs,
#   with date, time, and room/location for each.
#
#   The meeting app (studio.m-anage.com) loads its program from a JSON API:
#       Program -> Session -> Paper   (each Paper record has authors, time, location)
#   1. Open the app once in headless Chromium (so API requests are same-origin).
#   2. Crawl Home -> Program -> Session -> Paper, cache every paper to disk.
#   3. Match each paper's author list against your PI roster, write one CSV
#      + print a per-person schedule summary.
# ============================================================
!pip install -q pandas playwright nest_asyncio beautifulsoup4
!playwright install --with-deps chromium

import asyncio
import json
import os
import re
import time

import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

nest_asyncio.apply()  # lets asyncio.run() work inside Colab/Jupyter

# ---- Configuration ----
APP_URL = "https://studio.m-anage.com/agu/agu26/meetingapp.cgi/Paper/2085423"  # any app page works
API_BASE = "https://studio.m-anage.com/agu/agu26/meetingapi.cgi/"
PAPER_PAGE = "https://studio.m-anage.com/agu/agu26/meetingapp.cgi/Paper/"

# @ your PI roster — (full name, [emails]) just for reference/output; matching uses the name
PIS = [
    ("John Doe", ["email@gmail.edu"]),
    ("Jane Doe", ["email_2@gmail.edu"]),

]

HEADLESS = True
CONCURRENCY = 8          # simultaneous API requests inside the browser
CHUNK = 200              # paths per browser call
PROGRAM_ID_RANGE = range(5300, 5460)   # fallback seeds in case Home does not link to every Program
CACHE_FILE = "agu26_program_cache.json"   # delete to force a fresh download
OUT_CSV = "agu26_pi_presentations.csv"


# ============================================================
# Name matching: require first name AND last name inside the SAME author
# entry (not just anywhere in the full author-list string), to avoid
# false positives on common surnames (Kim, Li, Liu, Sun, Xie, ...).
# ============================================================
def name_parts(full_name):
    toks = full_name.replace("-", " ").split()
    return toks[0], toks[-1]  # first token, last token


PI_PATTERNS = []
for full_name, emails in PIS:
    first, last = name_parts(full_name)
    pat = re.compile(
        r"\b" + re.escape(first) + r"\b.*\b" + re.escape(last) + r"\b",
        re.IGNORECASE,
    )
    PI_PATTERNS.append((full_name, emails, pat))


def split_authors(author_string):
    # author_string looks like "John Smith, Jane Doe and Bob Lee"
    parts = re.split(r"\s*,\s*|\s+and\s+", author_string or "")
    return [p.strip() for p in parts if p.strip()]


def matched_pis(author_string):
    hits = []
    for entry in split_authors(author_string):
        for full_name, emails, pat in PI_PATTERNS:
            if pat.search(entry):
                hits.append(full_name)
    return sorted(set(hits))


# ============================================================
# Browser-side fetch (same-origin, so no CORS / cookie problems)
# ============================================================
FETCH_JS = """
async ([base, paths, conc]) => {
  const keep = ['Title','Abstract','FinalNumber','FinalPaperNumber','SessionTitle','Session_GoodType',
                'ProgramTitle','ProgramCode','SortableTimeString','TopDisplay','Withdrawn','PresentationLocation',
                'ParentList_ParentEntries'];
  const out = {};
  let i = 0;
  async function worker() {
    while (i < paths.length) {
      const p = paths[i++];
      for (let a = 0; a < 3; a++) {
        try {
          const r = await fetch(base + p, {credentials: 'same-origin'});
          if (r.ok) {
            let j = await r.json();
            if (p.startsWith('Paper/') && j && !Array.isArray(j)) {
              const t = {id: j.id};
              keep.forEach(k => { if (k in j) t[k] = j[k]; });
              j = t;
            }
            out[p] = j; break;
          }
          if (r.status === 404) { out[p] = null; break; }
        } catch (e) {}
        await new Promise(res => setTimeout(res, 600 * (a + 1)));
      }
      if (!(p in out)) out[p] = '__failed__';
    }
  }
  await Promise.all(Array.from({length: conc}, worker));
  return out;
}
"""


async def make_fetcher(page):
    async def fetch_many(paths):
        results = {}
        for s in range(0, len(paths), CHUNK):
            part = paths[s:s + CHUNK]
            results.update(await page.evaluate(FETCH_JS, [API_BASE, part, CONCURRENCY]))
        return results
    return fetch_many


# ============================================================
# Crawl: Home -> (Program ->) Session -> Paper
# ============================================================
FOLLOW = re.compile(r"^(Program|Session|ModuleProgramBook|ModuleSessionsByDay|ListItem|Day)/")
PAPER_REF = re.compile(r"^(?:Paper/|SlotData/Paper)(\d+)")


def refs_in(record):
    """Yield string references stored in a record's ChildList_* fields."""
    if not isinstance(record, dict):
        return
    for k, v in record.items():
        if k.startswith("ChildList_") and isinstance(v, list):
            for x in v:
                if isinstance(x, str):
                    yield x


async def crawl(fetch_many, verbose=True):
    """Return (set of paper ids, dict of visited-record counts by type)."""
    queue = ["Home/0"] + [f"Program/{i}" for i in PROGRAM_ID_RANGE]
    seen, paper_ids, kinds = set(queue), set(), {}
    while queue:
        batch, queue = queue[:CHUNK * 2], queue[CHUNK * 2:]
        res = await fetch_many(batch)
        for path, rec in res.items():
            if not isinstance(rec, dict) or not rec:
                continue
            kinds[path.split("/")[0]] = kinds.get(path.split("/")[0], 0) + 1
            for ref in refs_in(rec):
                m = PAPER_REF.match(ref)
                if m:
                    paper_ids.add(m.group(1))
                elif FOLLOW.match(ref) and ref not in seen:
                    seen.add(ref)
                    queue.append(ref)
        if verbose:
            print(f"  crawled {len(seen) - len(queue)} records | queue {len(queue)} | papers found {len(paper_ids)}")
    return paper_ids, kinds


async def download_papers(fetch_many, paper_ids, cache):
    todo = sorted(p for p in paper_ids if p not in cache)
    print(f"Papers to download: {len(todo)} (cached: {len(paper_ids) - len(todo)})")
    t0 = time.time()
    for s in range(0, len(todo), CHUNK):
        part = todo[s:s + CHUNK]
        res = await fetch_many([f"Paper/{p}" for p in part])
        for path, rec in res.items():
            if isinstance(rec, dict) and rec.get("Title") is not None:
                cache[path.split("/")[1]] = rec
        print(f"  [{min(s + CHUNK, len(todo))}/{len(todo)}] {time.time() - t0:.0f}s")
        json.dump(cache, open(CACHE_FILE, "w"))   # checkpoint
    return cache


async def build_cache():
    cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS)
        page = await browser.new_page()
        await page.goto(APP_URL, wait_until="networkidle")
        await asyncio.sleep(3)
        fetch_many = await make_fetcher(page)
        if not cache:
            print("Crawling the program tree ...")
            paper_ids, kinds = await crawl(fetch_many)
            print(f"Records visited by type: {kinds}")
            print(f"Unique papers discovered: {len(paper_ids)}")
            if len(paper_ids) < 1000:
                print("WARNING: far fewer papers than expected (AGU has ~20k+). The crawl did not reach every "
                      "session - widen PROGRAM_ID_RANGE.")
            cache = await download_papers(fetch_many, paper_ids, cache)
        await browser.close()
    return cache


# ============================================================
# Matching + CSV
# ============================================================
def clean_html(html):
    soup = BeautifulSoup(html or "", "html.parser")
    return re.sub(r"\s+", " ", soup.get_text(" ", strip=True))


def authors_from_topdisplay(html):
    soup = BeautifulSoup(html or "", "html.parser")
    for s in soup.find_all("sup"):
        s.decompose()
    text = re.sub(r"\s+", " ", soup.get_text(" ", strip=True))
    text = re.split(r"\(1\)", text)[0]          # drop the numbered affiliation list
    return re.sub(r"\s+,", ",", text).strip(" ,")


def paper_to_row(pid, rec):
    authors = authors_from_topdisplay(rec.get("TopDisplay"))
    hits = matched_pis(authors)
    if not hits:
        return None
    st = (rec.get("SortableTimeString") or "").split("_")      # 2026-12-08_17:05:00_17:15:00
    return {
        "matched_pis": ", ".join(hits),
        "title": clean_html(rec.get("Title")),
        "authors": authors,
        "presentation_type": rec.get("Session_GoodType", ""),
        "paper_number": rec.get("FinalPaperNumber") or rec.get("FinalNumber", ""),
        "main_session": clean_html(rec.get("SessionTitle")),
        "program": rec.get("ProgramTitle", ""),
        "date": st[0] if st else "",
        "time": f"{st[1][:5]}-{st[2][:5]}" if len(st) >= 3 else "",
        "presentation_location": rec.get("PresentationLocation", ""),
        "withdrawn": rec.get("Withdrawn", ""),
        "paper_id": pid,
        "url": f"{PAPER_PAGE}{pid}",
    }


def make_csv(cache):
    rows = [r for pid, rec in cache.items() if (r := paper_to_row(pid, rec))]
    df = pd.DataFrame(rows)
    if df.empty:
        print("No matches found for any PI in the roster.")
        return df
    df = df.sort_values(["date", "time"])
    df.to_csv(OUT_CSV, index=False)
    print(f"Papers searched: {len(cache)} | matching presentations: {len(df)} -> {OUT_CSV}")
    return df


def print_summary(df):
    print("\n" + "=" * 70)
    print("AGU26 SCHEDULE SUMMARY BY PI")
    print("=" * 70)
    found_names = set()
    for row in df.itertuples():
        for n in row.matched_pis.split(", "):
            found_names.add(n)
    for full_name, emails in PIS:
        sub = df[df["matched_pis"].str.contains(re.escape(full_name))]
        print(f"\n--- {full_name} ({emails[0]}) ---")
        if sub.empty:
            print("  No matching presentations found.")
            continue
        for row in sub.itertuples():
            print(f"  {row.date}  {row.time}  |  {row.presentation_location or 'location TBD'}")
            print(f"    \"{row.title}\"")
            print(f"    Session: {row.main_session}  |  {row.url}")


if __name__ == "__main__":
    cache = asyncio.run(build_cache())
    df = make_csv(cache)
    if len(df):
        print_summary(df)
    df

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:8 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,471 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,211 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,557 kB]
Get:13 https://r2u.stat.il